In [1]:
import os
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

import json, cv2, pytesseract

from src.preprocessing.preprocess import Preprocessing
from ocr.tesseract import TesseractModel, TesseractOCR
from ocr.paddle import Paddle, PaddleModel, PaddleParams
from src.preprocessing.preprocess import Preprocessing

In [2]:
FILE_PATHS = ["data/input/preprocess/1.png", "data/input/preprocess/2.png"]
OUT_DIR = "data/output/test"
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
os.makedirs(OUT_DIR, exist_ok=True)
cfg = json.load(open("config.json")) if os.path.exists("config.json") else {}

In [3]:
prep = Preprocessing(config=cfg)
tesseract = TesseractOCR(model=TesseractModel())
paddle = Paddle(model=PaddleModel(params=PaddleParams(
    lang='en', use_textline_orientation=True, device='cpu'
)))

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kunnic\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
c:\Users\kunnic\Desktop\code\VNDigitizeComprehensiveSystem_Team03\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kunnic\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kunnic\.paddlex\

In [4]:
prep_images = []
for path in FILE_PATHS:
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not read {path}")
    prep_img = prep._process(img).image
    cv2.imwrite(os.path.join(OUT_DIR, f"prep_{os.path.basename(path)}"), prep_img)
    prep_images.append(prep_img)

In [7]:
import time

tess_start = time.perf_counter()
tess_results = tesseract.recognize(prep_images)
tess_end = time.perf_counter()

tess_total_time = tess_end - tess_start
print(f"Tesseract Batch Time : {tess_total_time:.4f} seconds")

padd_start = time.perf_counter()
padd_results = paddle.recognize(prep_images)
padd_end = time.perf_counter()

padd_total_time = padd_end - padd_start
print(f"PaddleOCR Batch Time : {padd_total_time:.4f} seconds")
print("-" * 40)

Tesseract Batch Time : 2.0352 seconds

--- KIỂM TRA BIẾN RAW ---
1. Kiểu dữ liệu của raw: <class 'list'>
2. Số lượng kết quả bên trong raw: 1

--- KIỂM TRA PHẦN TỬ ĐẦU TIÊN TRONG RAW ---
3. Kiểu dữ liệu của phần tử: <class 'paddlex.inference.pipelines.ocr.result.OCRResult'>
4. Các phương thức/thuộc tính có thể gọi: ['clear', 'copy', 'fromkeys', 'get', 'img', 'items', 'json', 'keys', 'pop', 'popitem', 'print', 'save_all', 'save_to_img', 'save_to_json', 'setdefault', 'str', 'update', 'values']
5. Dữ liệu thực tế (dạng JSON): {'res': {'input_path': None, 'page_index': None, 'model_settings': {'use_doc_preprocessor': False, 'use_textline_orientation': True}, 'dt_polys': [[[97, 0], [445, 3], [445, 47], [97, 44]], [[596, 0], [1355, 1], [1355, 41], [596, 39]], [[190, 51], [352, 51], [352, 97], [190, 97]], [[713, 42], [1238, 46], [1238, 93], [712, 89]], [[45, 102], [496, 105], [496, 154], [45, 150]], [[111, 167], [427, 164], [427, 213], [111, 215]], [[0, 217], [549, 222], [548, 270], [0, 265]]

In [6]:
for path, prep_img, p_res, t_res in zip(FILE_PATHS, prep_images, padd_results, tess_results):
    base_name = os.path.basename(path)
    print(f"\n" + "="*50)
    print(f" FILE: {base_name} ")
    print("="*50)

    # -----------------------------------------
    # 1. PADDLE OCR (Green Boxes)
    # -----------------------------------------
    vis_paddle = prep_img.copy()
    if not isinstance(p_res.texts, str):
        for b in p_res.texts:
            x, y, w, h = b.bounding_box()
            cv2.rectangle(vis_paddle, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(vis_paddle, f"{b.confidence:.2f}", (x, max(y-5, 10)), 0, 0.4, (0, 0, 255), 1)

    cv2.imwrite(os.path.join(OUT_DIR, f"vis_paddle_{base_name}"), vis_paddle)
    print("\n>>> PADDLE RESULT <<<")
    print(p_res)

    # -----------------------------------------
    # 2. TESSERACT OCR (Blue Boxes)
    # -----------------------------------------
    vis_tess = prep_img.copy()
    if not isinstance(t_res.texts, str):
        for b in t_res.texts:
            x, y, w, h = b.bounding_box()
            cv2.rectangle(vis_tess, (x, y), (x+w, y+h), (255, 0, 0), 2) # Blue color
            cv2.putText(vis_tess, f"{b.confidence:.2f}", (x, max(y-5, 10)), 0, 0.4, (0, 0, 255), 1)

    cv2.imwrite(os.path.join(OUT_DIR, f"vis_tess_{base_name}"), vis_tess)
    print("\n>>> TESSERACT RESULT <<<")
    print(t_res)


 FILE: 1.png 

>>> PADDLE RESULT <<<
 OCR INFERENCE RESULT 
 Total Blocks : 39
 Avg Conf     : 0.9657
--------------------------------------------------
 [00] [Conf: 0.9484] "TOA ÁN NHAN DAN" | Box: (97, 0, 348, 47)
 [01] [Conf: 0.9707] "CQNG HOA XA HQI CHU NGHIA VIET NAM" | Box: (596, 0, 759, 41)
 [02] [Conf: 0.9957] "TINH BN" | Box: (190, 51, 162, 46)
 [03] [Conf: 0.9603] "Dc lp - Tyr do - Hanh phúc" | Box: (712, 42, 526, 51)
 [04] [Conf: 0.9435] "Bân án só: 03/2022/DSST" | Box: (45, 102, 451, 52)
 [05] [Conf: 0.9660] "Ngày: 23/11/2022" | Box: (111, 164, 316, 51)
 [06] [Conf: 0.9412] "V/v: Tranh châp vê kin dòi tài san" | Box: (0, 217, 549, 53)
 [07] [Conf: 0.9879] "NHAN DANH" | Box: (556, 312, 269, 54)
 [08] [Conf: 0.9469] "NUÓC CONG HòA XA HOI CHU NGHIA VIET NAM" | Box: (177, 359, 1029, 58)
 [09] [Conf: 0.9609] "TOA ÁN NHAN DAN TINH BN" | Box: (392, 449, 595, 55)
 [10] [Conf: 0.9358] "- Thành phân Hi dòng xét xi so thm gò..." | Box: (110, 550, 838, 60)
 [11] [Conf: 0.9648] "Thm ph